In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

GPU available: False


AssertionError: Torch not compiled with CUDA enabled

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os

base_path = "/content/drive/MyDrive/traffic_project"
os.makedirs(base_path + "/videos",  exist_ok=True)
os.makedirs(base_path + "/outputs", exist_ok=True)
print("Folders ready!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready!


In [5]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

data_path = "/content/drive/MyDrive/traffic_project/Accident_dataset/data"

IMG_SIZE = 224
BATCH_SIZE = 16

# Training transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Validation transforms
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Load datasets
train_data = datasets.ImageFolder(
    os.path.join(data_path, "train"),
    transform=train_transform
)

val_data = datasets.ImageFolder(
    os.path.join(data_path, "val"),
    transform=val_transform
)

test_data = datasets.ImageFolder(
    os.path.join(data_path, "test"),
    transform=val_transform
)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_data.classes

print("✅ Classes:", class_names)
print("📁 Train Images:", len(train_data))
print("📁 Val Images:", len(val_data))
print("📁 Test Images:", len(test_data))

✅ Classes: ['Accident', 'Non Accident']
📁 Train Images: 839
📁 Val Images: 98
📁 Test Images: 100


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "efficientnet_b0",
    pretrained=False,
    num_classes=2
)

model.load_state_dict(torch.load(
    "/content/drive/MyDrive/traffic_project/efficientnet_accident_best.pth",
    map_location=device
))

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

best_acc = 0
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs,1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/traffic_project/efficientnet_accident_best_v2.pth"
        )
        print("✅ Saved Improved Model")

Epoch 1: 91.84%
✅ Saved Improved Model
Epoch 2: 92.86%
✅ Saved Improved Model
Epoch 3: 92.86%
Epoch 4: 93.88%
✅ Saved Improved Model
Epoch 5: 94.90%
✅ Saved Improved Model


In [8]:
import json

mapping = {
    "class_names": class_names,
    "class_to_idx": train_data.class_to_idx
}

with open("/content/drive/MyDrive/traffic_project/class_mapping.json", "w") as f:
    json.dump(mapping, f)

print("✅ Class mapping saved")

✅ Class mapping saved


In [4]:
!pip install -q gradio timm opencv-python-headless ultralytics

import gradio as gr
import torch
import timm
import cv2
import json
import tempfile
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import transforms
from collections import deque
import gc
from ultralytics import YOLO

# ============================================================
# CONFIG
# ============================================================
BASE       = "/content/drive/MyDrive/traffic_project"
MODEL_PATH = f"{BASE}/efficientnet_accident_best_v2.pth"
MAP_PATH   = f"{BASE}/class_mapping.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ============================================================
# LOAD CLASSES
# ============================================================
with open(MAP_PATH, "r") as f:
    mapping = json.load(f)

class_names = mapping["class_names"]
num_classes = len(class_names)

ACCIDENT_IDX = next(
    (i for i, c in enumerate(class_names) if "accident" in c.lower()),
    0
)

print("Classes:", class_names)
print("Accident idx:", ACCIDENT_IDX)

# ============================================================
# LOAD CLASSIFIER
# ============================================================
model = timm.create_model(
    "efficientnet_b0",
    pretrained=False,
    num_classes=num_classes
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# ============================================================
# LOAD YOLO
# ============================================================
yolo = YOLO("yolov8n.pt")   # pretrained

# vehicle classes in COCO:
# car=2, motorcycle=3, bus=5, truck=7
VEHICLE_IDS = {2, 3, 5, 7}

print("✅ EfficientNet + YOLO loaded")

# ============================================================
# TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

# ============================================================
# GRAPH
# ============================================================
def make_graph(logs, threshold):
    if not logs:
        return None

    frames = [x[0] for x in logs]
    confs  = [x[1] for x in logs]

    plt.figure(figsize=(12,4))
    plt.plot(frames, confs, linewidth=2)
    plt.axhline(y=threshold, linestyle="--")
    plt.title("Accident Confidence Over Time")
    plt.xlabel("Frame")
    plt.ylabel("Probability")
    plt.grid(True)

    path = tempfile.mktemp(suffix=".png")
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path

# ============================================================
# HTML BOXES
# ============================================================
def success_box(msg):
    return f"""
    <div style="background:#16a34a;padding:18px;
    color:white;border-radius:12px;text-align:center;
    font-size:24px;font-weight:800;">{msg}</div>
    """

def danger_box(msg):
    return f"""
    <div style="background:#dc2626;padding:18px;
    color:white;border-radius:12px;text-align:center;
    font-size:24px;font-weight:800;">{msg}</div>
    """

def warning_box(msg):
    return f"""
    <div style="background:#92400e;padding:18px;
    color:white;border-radius:12px;text-align:center;
    font-size:22px;font-weight:700;">{msg}</div>
    """

def waiting_box():
    return success_box("⏳ Waiting for video analysis...")

# ============================================================
# YOLO VEHICLE CHECK
# ============================================================
def has_vehicle(frame):
    results = yolo.predict(
        source=frame,
        verbose=False,
        conf=0.25,
        imgsz=640
    )

    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return False

    cls_ids = boxes.cls.cpu().numpy().astype(int)

    for cid in cls_ids:
        if cid in VEHICLE_IDS:
            return True
    return False

# ============================================================
# DETECT
# ============================================================
def detect(video, threshold, buffer_size, min_streak, margin_required):
    try:
        if video is None:
            return warning_box("Upload a video first"), None, None, ""

        if isinstance(video, dict):
            video = video["name"]

        cap = cv2.VideoCapture(video)
        if not cap.isOpened():
            return warning_box("Cannot open video"), None, None, ""

        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps    = cap.get(cv2.CAP_PROP_FPS)

        if fps <= 0:
            fps = 20

        out_path = tempfile.mktemp(suffix=".mp4")
        writer = cv2.VideoWriter(
            out_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width,height)
        )

        prob_buffer  = deque(maxlen=int(buffer_size))
        label_buffer = deque(maxlen=int(buffer_size))

        logs = []
        total = 0
        accident_frames = 0
        frame_id = 0
        streak = 0
        SKIP = 2

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            total += 1
            frame_id += 1

            if frame_id % SKIP != 0:
                writer.write(frame)
                continue

            # ====================================================
            # STAGE 1: YOLO VEHICLE DETECTION
            # ====================================================
            vehicle_present = has_vehicle(frame)

            if not vehicle_present:
                decision = "SAFE"
                smooth_prob = 0.0
                color = (0,200,0)
                streak = 0

            else:
                # =================================================
                # STAGE 2: CLASSIFICATION
                # =================================================
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                x = transform(Image.fromarray(rgb)).unsqueeze(0).to(device)

                with torch.no_grad():
                    probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()

                accident_prob = float(probs[ACCIDENT_IDX])
                pred_class = int(np.argmax(probs))
                margin = float(probs[ACCIDENT_IDX] - probs[1-ACCIDENT_IDX])

                prob_buffer.append(accident_prob)
                label_buffer.append(pred_class)

                smooth_prob = float(np.mean(prob_buffer))

                majority_accident = (
                    list(label_buffer).count(ACCIDENT_IDX) >
                    len(label_buffer) * 0.6
                )

                if smooth_prob < threshold:
                    decision = "UNCERTAIN"
                    color = (0,255,255)
                    streak = 0

                elif (
                    smooth_prob >= threshold and
                    margin >= margin_required and
                    majority_accident
                ):
                    streak += 1

                    if streak >= int(min_streak):
                        decision = "ACCIDENT"
                        color = (0,0,255)

                        if streak == int(min_streak):
                            accident_frames += 1
                    else:
                        decision = "VERIFYING"
                        color = (0,165,255)

                else:
                    decision = "SAFE"
                    color = (0,200,0)
                    streak = 0

            logs.append((frame_id, smooth_prob, decision))

            # ====================================================
            # DRAW
            # ====================================================
            cv2.rectangle(frame, (0,0), (width,60), (0,0,0), -1)

            txt = {
                "ACCIDENT":"ACCIDENT DETECTED",
                "VERIFYING":"VERIFYING...",
                "SAFE":"NO ACCIDENT",
                "UNCERTAIN":"UNCERTAIN"
            }[decision]

            cv2.putText(
                frame,
                f"{txt} ({smooth_prob:.0%})",
                (15,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                color,
                2
            )

            writer.write(frame)

        cap.release()
        writer.release()

        graph = make_graph(logs, threshold)

        summary = f"""
Total Frames      : {total}
Accident Events   : {accident_frames}
Model 1           : YOLOv8n (Vehicle Detection)
Model 2           : EfficientNet-B0 (Accident Classification)
Temporal Module   : Buffer + Streak Logic
Device            : {device}
"""

        if accident_frames > 0:
            alert = danger_box(f"🚨 ACCIDENT DETECTED ({accident_frames} confirmed event(s))")
        else:
            alert = success_box("✅ NO ACCIDENT DETECTED")

        return alert, out_path, graph, summary

    except Exception as e:
        return warning_box(str(e)), None, None, ""

# ============================================================
# UI
# ============================================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🚗 Smart Traffic Accident Detection")
    gr.Markdown("### Multi-Model Pipeline: YOLOv8 + EfficientNet + Temporal Validation")

    alert_html = gr.HTML(waiting_box())

    with gr.Row():

        with gr.Column(scale=1):
            video_in = gr.Video(label="Upload Video")

            threshold_sl = gr.Slider(0.50,0.99,0.85,0.01,label="Confidence Threshold")
            margin_sl    = gr.Slider(0.05,0.60,0.30,0.05,label="Margin Required")
            streak_sl    = gr.Slider(2,15,6,1,label="Min Streak")
            buffer_sl    = gr.Slider(3,20,10,1,label="Temporal Buffer")

            run_btn = gr.Button("🚀 Run Detection")
            clear_btn = gr.Button("🗑 Clear")

        with gr.Column(scale=2):
            video_out = gr.Video(label="Processed Output")
            graph_out = gr.Image(label="Confidence Graph")
            summary_out = gr.Textbox(lines=10,label="Summary")

    run_btn.click(
        detect,
        inputs=[video_in, threshold_sl, buffer_sl, streak_sl, margin_sl],
        outputs=[alert_html, video_out, graph_out, summary_out]
    )

    clear_btn.click(
        lambda:(waiting_box(), None, None, ""),
        outputs=[alert_html, video_out, graph_out, summary_out]
    )

demo.launch(share=True, debug=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using: cpu
Classes: ['Accident', 'Non Accident']
Accident idx: 0
✅ EfficientNet + YOLO loaded


/tmp/ipykernel_8499/3610248484.py:320: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://042c39eb29d9a68a97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
import cv2, os

video_path = "/content/drive/MyDrive/traffic_project/videos/no_accident_1.mp4"

save_dir = "/content/drive/MyDrive/traffic_project/Accident_dataset/data/train/Non Accident"
os.makedirs(save_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
saved = 0

for i in range(0, total, 15):   # save every 15th frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ret, frame = cap.read()

    if ret:
        path = os.path.join(save_dir, f"hardneg_truck_{i:04d}.jpg")
        cv2.imwrite(path, frame)
        saved += 1

cap.release()
print(f"✅ Saved {saved} hard negative frames")

✅ Saved 40 hard negative frames


In [5]:

# CELL 1 — INSTALL + IMPORTS + DATASET LOADER
# EfficientNet-B0 + LSTM Accident Detection

!pip install -q timm gradio opencv-python-headless

import os
import cv2
import json
import math
import random
import tempfile
import numpy as np
from PIL import Image
from glob import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
import timm

import matplotlib.pyplot as plt
import gradio as gr
from collections import deque

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE = "/content/drive/MyDrive/traffic_project"
DATA_PATH = f"{BASE}/Accident_dataset/data"

SEQ_LEN = 10
IMG_SIZE = 224
BATCH_SIZE = 4
NUM_CLASSES = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ------------------------------------------------------------
# TRANSFORMS
# ------------------------------------------------------------
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

# ------------------------------------------------------------
# BUILD SEQUENCES FROM IMAGE FOLDERS
# ------------------------------------------------------------
class SequenceFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None, seq_len=10):
        self.transform = transform
        self.seq_len = seq_len
        self.samples = []
        self.class_names = sorted(os.listdir(root_dir))

        for label_idx, cls in enumerate(self.class_names):
            cls_path = os.path.join(root_dir, cls)
            imgs = sorted(glob(os.path.join(cls_path, "*")))

            # make chunks of seq_len
            for i in range(0, len(imgs) - seq_len + 1, seq_len):
                seq = imgs[i:i+seq_len]
                self.samples.append((seq, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq_paths, label = self.samples[idx]
        frames = []

        for p in seq_paths:
            img = Image.open(p).convert("RGB")
            img = self.transform(img)
            frames.append(img)

        frames = torch.stack(frames)   # [T,C,H,W]
        return frames, label

# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------
train_ds = SequenceFolderDataset(
    os.path.join(DATA_PATH, "train"),
    transform=train_tf,
    seq_len=SEQ_LEN
)

val_ds = SequenceFolderDataset(
    os.path.join(DATA_PATH, "val"),
    transform=val_tf,
    seq_len=SEQ_LEN
)

test_ds = SequenceFolderDataset(
    os.path.join(DATA_PATH, "test"),
    transform=val_tf,
    seq_len=SEQ_LEN
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Train sequences:", len(train_ds))
print("Val sequences:", len(val_ds))
print("Test sequences:", len(test_ds))

Using: cpu
Classes: ['Accident', 'Non Accident']
Train sequences: 83
Val sequences: 9
Test sequences: 9


In [6]:
# ============================================================
# CELL 2 — MODEL: EfficientNet-B0 + LSTM
# ============================================================

import torch
import torch.nn as nn
import timm

class EfficientNetLSTM(nn.Module):
    def __init__(self, num_classes=2, hidden_size=128, num_layers=1, dropout=0.3):
        super(EfficientNetLSTM, self).__init__()

        # EfficientNet feature extractor
        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0   # remove classifier head
        )

        self.feature_dim = self.backbone.num_features

        # LSTM
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0 if num_layers == 1 else dropout
        )

        # Final classifier
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        x shape: [B, T, C, H, W]
        B = batch size
        T = sequence length
        """

        B, T, C, H, W = x.shape

        # reshape for CNN
        x = x.view(B * T, C, H, W)

        # EfficientNet features
        feats = self.backbone(x)   # [B*T, F]

        # reshape back to sequence
        feats = feats.view(B, T, -1)   # [B, T, F]

        # LSTM output
        out, _ = self.lstm(feats)

        # last timestep output
        out = out[:, -1, :]

        out = self.dropout(out)
        out = self.fc(out)

        return out


# ------------------------------------------------------------
# CREATE MODEL
# ------------------------------------------------------------
model = EfficientNetLSTM(
    num_classes=2,
    hidden_size=128,
    num_layers=1,
    dropout=0.3
).to(device)

print(model)
print("✅ EfficientNet + LSTM model created")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNetLSTM(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_p

In [8]:
# ============================================================
# CELL 3 — TRAINING + VALIDATION + SAVE BEST MODEL (FIXED)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

EPOCHS = 10
best_acc = 0.0
SAVE_PATH = f"{BASE}/efficientnet_lstm_best.pth"

for epoch in range(EPOCHS):

    # ---------------- TRAIN ----------------
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for sequences, labels in pbar:
        sequences = sequences.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(sequences)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100*train_correct/train_total:.2f}%"
        })

    train_acc = 100 * train_correct / train_total

    # ---------------- VALIDATION ----------------
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            outputs = model(sequences)
            _, preds = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    scheduler.step(val_acc)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Acc: {train_acc:.2f}%")
    print(f"Val Acc  : {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print("✅ Best model saved")

print("\n🎉 Training Completed")
print(f"🏆 Best Validation Accuracy: {best_acc:.2f}%")

Epoch 1/10: 100%|██████████| 21/21 [08:34<00:00, 24.50s/it, loss=0.6993, acc=49.40%]



Epoch 1/10
Train Acc: 49.40%
Val Acc  : 66.67%
✅ Best model saved


Epoch 2/10: 100%|██████████| 21/21 [04:32<00:00, 12.98s/it, loss=0.5630, acc=68.67%]



Epoch 2/10
Train Acc: 68.67%
Val Acc  : 66.67%


Epoch 3/10: 100%|██████████| 21/21 [04:42<00:00, 13.47s/it, loss=0.7357, acc=73.49%]


...


Epoch 8/10
Train Acc: 86.75%
Val Acc  : 100.00%


Epoch 9/10: 100%|██████████| 21/21 [04:38<00:00, 13.25s/it, loss=0.3904, acc=95.18%]



Epoch 9/10
Train Acc: 95.18%
Val Acc  : 100.00%


Epoch 10/10: 100%|██████████| 21/21 [04:36<00:00, 13.15s/it, loss=0.1979, acc=92.77%]



Epoch 10/10
Train Acc: 92.77%
Val Acc  : 88.89%

🎉 Training Completed
🏆 Best Validation Accuracy: 100.00%


In [ ]:

# Smart Traffic Accident Detection
# EfficientNet + LSTM + Premium UI + Alarm


!pip -q install gradio opencv-python-headless matplotlib timm

import os, cv2, json, torch, tempfile, numpy as np, gradio as gr
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque
from torchvision import transforms
import timm
from PIL import Image

# ----------------------------
# CONFIG
# ----------------------------
MODEL_PATH = "/content/drive/MyDrive/traffic_project/efficientnet_lstm_best.pth"
MAP_PATH   = "/content/drive/MyDrive/traffic_project/class_mapping.json"

IMG_SIZE = 224
SEQ_LEN  = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)

# ----------------------------
# LOAD CLASS MAPPING
# ----------------------------
with open(MAP_PATH, "r") as f:
    mapping = json.load(f)

class_names = mapping["class_names"]
ACCIDENT_IDX = mapping["class_to_idx"]["Accident"]

# ----------------------------
# TRANSFORM
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ----------------------------
# MODEL
# ----------------------------
class EfficientNetLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=False,
            num_classes=0
        )

        self.feature_dim = self.backbone.num_features

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=128,   # ✅ match trained model
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128, num_classes)   # ✅ match trained model

    def forward(self, x):
        B,T,C,H,W = x.shape
        x = x.view(B*T,C,H,W)

        feat = self.backbone(x)
        feat = feat.view(B,T,-1)

        out,_ = self.lstm(feat)
        out = out[:,-1,:]

        out = self.dropout(out)
        out = self.fc(out)

        return out

model = EfficientNetLSTM(num_classes=2)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print("✅ Model Loaded")

# ----------------------------
# CSS PREMIUM UI
# ----------------------------
CSS = """
body{
    background: linear-gradient(135deg,#050816,#0f172a,#111827);
}
.gradio-container{
    font-family: 'Inter', sans-serif;
    color:white;
}
.main-title{
    text-align:center;
    font-size:38px;
    font-weight:800;
    background:linear-gradient(90deg,#ff416c,#ff4b2b,#7f5af0);
    -webkit-background-clip:text;
    -webkit-text-fill-color:transparent;
    margin-bottom:8px;
}
.sub-title{
    text-align:center;
    color:#cbd5e1;
    font-size:16px;
    margin-bottom:20px;
}
.card{
    background:rgba(255,255,255,0.06);
    border:1px solid rgba(255,255,255,0.08);
    backdrop-filter: blur(12px);
    border-radius:18px;
    padding:14px;
}
button{
    border-radius:14px !important;
    font-weight:700 !important;
}
"""

# ----------------------------
# GRAPH
# ----------------------------
def make_graph(vals, threshold):
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(vals, linewidth=2)
    ax.axhline(threshold, linestyle="--")
    ax.set_title("Accident Confidence Over Time")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Probability")
    plt.tight_layout()
    return fig

# ----------------------------
# ALARM HTML
# ----------------------------
def alarm_html(trigger=False):
    if not trigger:
        return "<div style='padding:15px;color:gray;'>No Alarm</div>"

    return """
    <div style="
        background:linear-gradient(90deg,#ff0000,#ff5e5e);
        color:white;
        padding:20px;
        border-radius:15px;
        text-align:center;
        font-size:28px;
        font-weight:900;
        animation: blink 1s infinite;
    ">
    🚨 ACCIDENT DETECTED 🚨
    <audio autoplay>
        <source src="https://actions.google.com/sounds/v1/alarms/alarm_clock.ogg" type="audio/ogg">
    </audio>
    </div>

    <style>
    @keyframes blink{
        0%{opacity:1;}
        50%{opacity:0.4;}
        100%{opacity:1;}
    }
    </style>
    """

# ----------------------------
# DETECT FUNCTION
# ----------------------------
def detect(video, threshold):
    if video is None:
        return None, "Upload Video", None, "No Summary", "<div>No Alarm</div>"

    cap = cv2.VideoCapture(video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 20

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    temp_out = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4").name
    writer = cv2.VideoWriter(
        temp_out,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width,height)
    )

    seq = deque(maxlen=SEQ_LEN)
    probs = []
    accident_events = 0
    accident_triggered = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb)
        tensor = transform(img)
        seq.append(tensor)

        label = "Collecting..."
        p = 0

        if len(seq) == SEQ_LEN:
            x = torch.stack(list(seq)).unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(x)
                prob = torch.softmax(out, dim=1)[0]
                p = float(prob[ACCIDENT_IDX])

            probs.append(p)

            if p >= threshold:
                label = f"🚨 ACCIDENT ({p:.2f})"
                color = (0,0,255)
                accident_events += 1
                accident_triggered = True
            else:
                label = f"✅ SAFE ({p:.2f})"
                color = (0,255,0)

            cv2.putText(frame,label,(20,45),
                        cv2.FONT_HERSHEY_SIMPLEX,1,color,3)

        writer.write(frame)

    cap.release()
    writer.release()

    graph = make_graph(probs, threshold)

    prediction = "🚨 Accident Detected!" if accident_triggered else "✅ No Accident"

    summary = f"""
Total Frames      : {total_frames}
Accident Events   : {accident_events}
Model             : EfficientNet-B0 + LSTM
Sequence Length   : {SEQ_LEN}
Device            : {device}
"""

    return temp_out, prediction, graph, summary, alarm_html(accident_triggered)

# ----------------------------
# UI
# ----------------------------
with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:

    gr.HTML("<div class='main-title'>🚗 Smart Traffic Accident Detection</div>")
    gr.HTML("<div class='sub-title'>EfficientNet-B0 + LSTM +  Dashboard</div>")

    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(label="📤 Upload CCTV Video")
            threshold = gr.Slider(
                0.5,0.99,value=0.8,step=0.01,
                label="Confidence Threshold"
            )
            run_btn = gr.Button("🚀 Run Detection", variant="primary")

        with gr.Column(scale=1):
            pred = gr.Textbox(label="Prediction")
            video_out = gr.Video(label="🎬 Processed Video")
            graph = gr.Plot(label="📈 Confidence Graph")
            summary = gr.Textbox(label="📋 Summary", lines=8)
            alarm_box = gr.HTML(label="🚨 Alarm")

    run_btn.click(
        detect,
        inputs=[video_in, threshold],
        outputs=[video_out, pred, graph, summary, alarm_box]
    )

demo.launch(debug=True, share=True)

Using: cpu
✅ Model Loaded


/tmp/ipykernel_8499/920298990.py:264: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_8499/920298990.py:264: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0649f74c5e0dea52a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error